In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor


In [2]:
df = pd.read_csv(
    r"E:\Personal Things\02 Projects\23 Ride hailing\Data Processing\yellow_tripdata_2015-01.csv"
)

df = df.sample(n=1_500_000, random_state=42)
df.shape


(1500000, 19)

In [3]:
df["pickup_time"] = pd.to_datetime(df["tpep_pickup_datetime"])


In [4]:
df["request_time"] = df["pickup_time"] - pd.to_timedelta(
    np.random.randint(2, 8, size=len(df)), unit="m"
)


In [5]:
df = df[
    (df["pickup_latitude"].between(40.5, 41.0)) &
    (df["pickup_longitude"].between(-74.3, -73.6))
]


In [6]:
# Create Pickup Zones (COARSE, for demand)
kmeans = KMeans(
    n_clusters=20,
    random_state=42,
    n_init=10
)

df["pickup_zone"] = kmeans.fit_predict(
    df[["pickup_latitude", "pickup_longitude"]]
)

df["pickup_zone"] = "Zone_" + df["pickup_zone"].astype(str)


In [7]:
# Time Bucketing (IMPORTANT)
df["time_window"] = df["request_time"].dt.floor("15min")


In [8]:
demand_df = (
    df.groupby(["pickup_zone", "time_window"])
      .size()
      .reset_index(name="demand")
)

demand_df.shape


(54716, 3)

In [9]:
demand_df["hour"] = demand_df["time_window"].dt.hour
demand_df["day"] = demand_df["time_window"].dt.day
demand_df["weekday"] = demand_df["time_window"].dt.weekday
demand_df["is_weekend"] = demand_df["weekday"].isin([5, 6]).astype(int)


In [10]:
zone_map = {z: i for i, z in enumerate(demand_df["pickup_zone"].unique())}
demand_df["zone_id"] = demand_df["pickup_zone"].map(zone_map)


In [11]:
demand_df = demand_df.sort_values(
    ["pickup_zone", "time_window"]
)

demand_df["demand_lag_1"] = (
    demand_df.groupby("pickup_zone")["demand"].shift(1)
)

demand_df["demand_lag_2"] = (
    demand_df.groupby("pickup_zone")["demand"].shift(2)
)

demand_df["demand_lag_4"] = (
    demand_df.groupby("pickup_zone")["demand"].shift(4)
)


In [12]:
demand_df = demand_df.dropna().reset_index(drop=True)
demand_df.shape


(54636, 11)

In [13]:
feature_cols = [
    "zone_id",
    "hour",
    "weekday",
    "is_weekend",
    "demand_lag_1",
    "demand_lag_2",
    "demand_lag_4"
]

X = demand_df[feature_cols]
y = demand_df["demand"]

X.shape, y.shape


((54636, 7), (54636,))

In [14]:
split_idx = int(len(demand_df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


In [15]:
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001803 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 479
[LightGBM] [Info] Number of data points in the train set: 43708, number of used features: 7
[LightGBM] [Info] Start training from score 30.003226


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=300,
              num_leaves=64, random_state=42, subsample=0.8)

In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

mae, rmse


(3.5374636421058465, 5.3871507159953325)

In [18]:
sample = X_test.iloc[[0]]

predicted_demand = model.predict(sample)
predicted_demand


array([2.01291648])

In [19]:
zone = demand_df.iloc[X_test.index[0]]["pickup_zone"]
time = demand_df.iloc[X_test.index[0]]["time_window"]

zone, time, predicted_demand


('Zone_6', Timestamp('2015-01-06 04:45:00'), array([2.01291648]))